# Extracción de Códigos a 10 Dígitos
## Fases de Tratado
### Fase 0: Configuración General

El objetivo de este script es procesar la base de datos de tarifas de Estados Unidos (HTS) para extraer específicamente las fracciones arancelarias a 10 dígitos. Implementa una lógica de "barrera" basada en la indentación para concatenar correctamente las descripciones padre-hijo sin sobrepasar los límites lógicos de las partidas.

Dependencias requeridas:
- `pandas (pd):` Manipulación de estructuras de datos.
- `numpy (np):` Operaciones numéricas y manejo de nulos.
- `os:` Manejo de rutas y creación de directorios.

Variables Globales:
- **Rutas:** Apuntan al archivo Raw (CSV) y definen la ruta de salida del nuevo CSV procesado a 10 dígitos.

In [1]:
import pandas as pd
import numpy as np
import os

PATH_HTS_CSV = "../data/raw/htsdata.csv"
PATH_SALIDA_10D = "../data/intermediate/diez_digitos.csv"

print("--- CONFIGURACIÓN CARGADA ---")
print(f"Input CSV: {PATH_HTS_CSV}")
print(f"Output:    {PATH_SALIDA_10D}")

### Fase 0.5: Definición de Funciones

Se definen las funciones necesarias separadas por su nivel de operación para facilitar el mantenimiento:

**1. Funciones Auxiliares (`_nombre`)**
Operaciones atómicas de limpieza de texto.

- **`_limpiar_descripcion`**: Recibe un texto descriptivo, elimina espacios en blanco y remueve los dos puntos (`:`) que suelen aparecer al final de descripciones padre en el HTS.

**2. Funciones Principales (`nombre`)**
Orquestadores de la lógica analítica.

- **`procesar_logica_barrera`**: Itera sobre la base HTS manteniendo una pila (`stack` o diccionario en memoria) de la jerarquía actual basada en la columna `Indent`. Cuando encuentra un código de 10 dígitos sin Tasa General, busca hacia arriba en la jerarquía para concatenar la descripción completa del producto, deteniéndose al encontrar una "barrera" (códigos de 4, 6 u 8 dígitos).

In [2]:
# --- FUNCIONES AUXILIARES ---

def _limpiar_descripcion(desc):
    """Limpia el texto de la descripción y quita dos puntos finales."""
    if pd.isna(desc) or not isinstance(desc, str):
        return ""
    desc = str(desc).strip()
    if desc.endswith(':'):
        desc = desc[:-1]
    return desc.strip()


# --- FUNCIONES PRINCIPALES ---

def procesar_logica_barrera(input_path):
    """
    Lee el CSV raw, aplica lógica de pila basada en indentación 
    y construye descripciones para códigos de 10 dígitos.
    """
    print(f">> Procesando {input_path}...")
    try:
        # Leemos todo como string para no perder ceros a la izquierda
        df = pd.read_csv(input_path, dtype=str)
    except FileNotFoundError:
        print("ERROR: No se encontró el archivo de entrada.")
        return pd.DataFrame()

    final_rows = []
    # La Pila (Stack) guarda: { Indent: { 'desc': texto, 'code_len': longitud_codigo } }
    hierarchy = {}

    for index, row in df.iterrows():
        # 1. PARSEO DE DATOS
        try:
            raw_indent = row.get('Indent')
            if pd.isna(raw_indent) or str(raw_indent).strip() == '':
                continue
            indent = int(float(str(raw_indent)))
        except ValueError:
            continue

        raw_code = str(row.get('HTS Number', '')) if pd.notna(row.get('HTS Number')) else ""
        clean_code = raw_code.replace(".", "").strip()
        if clean_code.lower() == 'nan': clean_code = ""
        code_len = len(clean_code)

        raw_desc = _limpiar_descripcion(row.get('Description', ''))
        general_rate = str(row.get('General Rate of Duty', '')).strip()
        if general_rate.lower() == 'nan': general_rate = ""

        # 2. GESTIÓN DE LA PILA
        # Borramos niveles más profundos o iguales al actual
        levels_to_remove = [k for k in hierarchy.keys() if k >= indent]
        for k in levels_to_remove:
            del hierarchy[k]
            
        hierarchy[indent] = {
            'desc': raw_desc,
            'code_len': code_len
        }

        # 3. PROCESAMIENTO DE CÓDIGOS A 10 DÍGITOS
        if code_len == 10:
            # Condición excluyente: Si tiene Tasa General, se descarta.
            if general_rate != "":
                continue

            desc_parts = []
            if hierarchy[indent]['desc']:
                desc_parts.insert(0, hierarchy[indent]['desc'])
            
            # Búsqueda hacia arriba (Ancestros)
            current_search_indent = indent - 1
            while current_search_indent >= 0:
                if current_search_indent in hierarchy:
                    node = hierarchy[current_search_indent]
                    node_code_len = node['code_len']
                    
                    # Barrera: Código de 4, 6 u 8 dígitos
                    if node_code_len in [4, 6, 8]:
                        break 
                    
                    if node['desc']:
                        desc_parts.insert(0, node['desc'])
                
                current_search_indent -= 1
            
            full_desc = ". ".join(desc_parts) + "."
            final_rows.append({
                'HTS Code': clean_code,
                'Description': full_desc
            })

    return pd.DataFrame(final_rows)

### Fase 1: Extracción y Limpieza Lógica
Se ejecuta el orquestador principal que lee el archivo RAW, filtra los códigos estadísticos de 10 dígitos y genera su descripción completa sin alterar la jerarquía arancelaria base.

In [3]:
DF_DIEZ_DIGITOS = procesar_logica_barrera(PATH_HTS_CSV)
print(f">> Proceso finalizado. Se generaron {len(DF_DIEZ_DIGITOS)} registros limpios.")

### Fase 2: Exportación Final
El DataFrame procesado se exporta a la ruta intermedia designada en formato CSV, asegurando la creación de los directorios de ser necesario.

In [4]:
print(f"Generando CSV: {PATH_SALIDA_10D}...")

try:
    if not DF_DIEZ_DIGITOS.empty:
        os.makedirs(os.path.dirname(PATH_SALIDA_10D), exist_ok=True)
        DF_DIEZ_DIGITOS.to_csv(PATH_SALIDA_10D, index=False)
        print("¡ÉXITO! Archivo generado correctamente con estructura jerárquica.")
        print("\nEjemplo de salida:")
        print(DF_DIEZ_DIGITOS.head())
    else:
        print("ADVERTENCIA: No se generaron registros, archivo no exportado.")
except Exception as e:
    print(f"ERROR CRÍTICO EN EXPORTACIÓN: {e}")